In [46]:
import pandas as pd
import os
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score



df_encoded_path = os.path.join(os.getcwd(), '..', 'Data', 'processed', 'df.data')
df_encoded = pd.read_csv(df_encoded_path, sep=",")

df_encoded["num"] = (df_encoded["num"] > 0).astype(int)

X = df_encoded.drop("num", axis=1)
y = df_encoded["num"]

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

from sklearn.preprocessing import StandardScaler

numerical_features = ['age','trestbps','chol','thalach','oldpeak']

scaler = StandardScaler()
X_train[numerical_features] = scaler.fit_transform(X_train[numerical_features])
X_test[numerical_features] = scaler.transform(X_test[numerical_features])

from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)


y_pred = model.predict(X_test)

from sklearn.metrics import accuracy_score, classification_report

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

Accuracy: 0.7954545454545454
              precision    recall  f1-score   support

           0       0.80      0.72      0.76        39
           1       0.79      0.86      0.82        49

    accuracy                           0.80        88
   macro avg       0.80      0.79      0.79        88
weighted avg       0.80      0.80      0.79        88



In [47]:
rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=6,
    min_samples_split=5,
    min_samples_leaf=2,
    random_state=42
)

rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)

print("RF Accuracy:", accuracy_score(y_test, y_pred_rf))

RF Accuracy: 0.8295454545454546


In [48]:
from sklearn.ensemble import GradientBoostingClassifier

gb = GradientBoostingClassifier(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=3,
    random_state=42
)

gb.fit(X_train, y_train)

y_pred_gb = gb.predict(X_test)

print("GB Accuracy:", accuracy_score(y_test, y_pred_gb))

GB Accuracy: 0.7727272727272727


In [49]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    "n_estimators": [100, 200, 300],
    "max_depth": [3, 5, 7],
    "min_samples_split": [2, 5],
}

grid = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid,
    cv=5,
    scoring="accuracy"
)

grid.fit(X_train, y_train)

print("Best params:", grid.best_params_)
print("Best score:", grid.best_score_)

Best params: {'max_depth': 3, 'min_samples_split': 2, 'n_estimators': 100}
Best score: 0.8308902691511388


In [50]:
from sklearn.model_selection import cross_val_score

scores = cross_val_score(rf, X, y, cv=5)
print("CV mean:", scores.mean())

CV mean: 0.810135841170324


In [51]:
import pandas as pd

importance = pd.Series(rf.feature_importances_, index=X.columns)
print(importance.sort_values(ascending=False))

thalach        0.138519
exang          0.127966
cp_4           0.122969
oldpeak        0.104409
cp_2           0.089452
age            0.081047
slope_2.0      0.073074
chol           0.066268
sex            0.050476
trestbps       0.045985
fbs            0.023516
cp_3           0.016369
thal_7.0       0.013780
thal_6.0       0.013255
restecg_1.0    0.012686
slope_3.0      0.006406
thal_3.0       0.005873
slope_1.0      0.004876
restecg_2.0    0.003028
thal_5.0       0.000047
thal_2.0       0.000000
thal_4.0       0.000000
ca_1.0         0.000000
ca_2.0         0.000000
ca_9.0         0.000000
dtype: float64
